# 03 Predict with M7

Scores every held-out station with M7, the deterministic threshold rule, through the public package. Nothing is fitted; the thresholds are configuration.

Abbreviations used here: **RPF** is reverse power flow, the condition where a distribution substation exports power because rooftop solar exceeds local demand; a *wrong RPF sign* is a meter recording that stores the export as an import. **M7** is the deterministic threshold rule, **M8** the two-stage XGBoost classifier and **M9** the compact counterfactual method (revision 2). **MW** and **MWh** are megawatts and megawatt-hours; one interval is 15 minutes.

**Inputs.** The fold manifest (notebook 01) and the two frozen datasets.

**Outputs.** `outputs/01_final_evaluation/03_m7/intervals_m7.parquet` (one row per quarter-hour of every complete held-out site-day: the flagged slots, the day flag and the package confidence) and `manifests/03_m7_predict.json`.

**Approximate runtime.** About two minutes.

**Prerequisites.** Notebook 01.

**Main process.**

1. Load the configuration and the folds.
2. For each held-out station, run `pynrpf.api.run_inference` with the M7 configuration and align the flags back to the input rows.
3. Write the interval prediction table in the schema shared by all methods.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display


def article_root() -> Path:
    """Locate publication/2_journal_article from JupyterLab, VS Code or the repository root."""
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate / "final_eval" / "cli.py").exists():
            return candidate
        nested = candidate / "publication" / "2_journal_article"
        if (nested / "final_eval" / "cli.py").exists():
            return nested
    raise FileNotFoundError("Could not locate publication/2_journal_article.")


ARTICLE = article_root()
sys.path.insert(0, str(ARTICLE))
sys.path.insert(0, str(ARTICLE.parents[1] / "src"))  # the repository's pynrpf package

from final_eval import cli, config  # noqa: E402

SETTINGS = config.load()  # verifies the frozen dataset hashes
OUT = SETTINGS.output_root()
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print("Article root:", ARTICLE.relative_to(ARTICLE.parents[2]))

## 2. Predict

M7 is applied station by station so that every prediction carries its fold id, exactly like M8 and M9.

In [ ]:
table = cli.stage_predict(SETTINGS, "m7")
by_day = table.groupby(["cohort", "station", "date"]).agg(day_flagged=("pred_interval", "any"), slots_flagged=("pred_interval", "sum")).reset_index()
summary = by_day.groupby(["cohort", "station"]).agg(days=("date", "size"), days_flagged=("day_flagged", "sum"), slots_flagged=("slots_flagged", "sum")).reset_index()
display(summary)

## Conclusion

The M7 interval table is written. Its site-day outcomes (a day is corrected when at least one slot is flagged) are derived in notebook 06.